In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

In [4]:
consumer = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-consDF.pqt")
consumer.head()

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
0,0,2021-09-01,726.0,0.0
1,1,2021-07-01,626.0,0.0
2,2,2021-05-01,680.0,0.0
3,3,2021-03-01,734.0,0.0
4,4,2021-10-01,676.0,0.0


In [5]:
account = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-acctDF.pqt")
account.head()

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,3023,0,SAVINGS,2021-08-31,90.57
1,3023,1,CHECKING,2021-08-31,225.95
2,4416,2,SAVINGS,2022-03-31,15157.17
3,4416,3,CHECKING,2022-03-31,66.42
4,4227,4,CHECKING,2021-07-31,7042.90


In [6]:
transaction = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-trxnDF.pqt")
transaction.head()

,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date
0,3023,0,4,0.05,CREDIT,2021-04-16
1,3023,1,12,481.56,CREDIT,2021-04-30
2,3023,2,4,0.05,CREDIT,2021-05-16
3,3023,3,4,0.07,CREDIT,2021-06-16
4,3023,4,4,0.06,CREDIT,2021-07-16


In [9]:
mapping = pd.read_csv("/uss/hdsi-prismdata/q2-ucsd-cat-map.csv")
mapping.head()

,category_id,category
0,0,SELF_TRANSFER
1,1,EXTERNAL_TRANSFER
2,2,DEPOSIT
3,3,PAYCHECK
4,4,MISCELLANEOUS


In [26]:
consumer["evaluation_date"] = pd.to_datetime(consumer["evaluation_date"])
transaction["posted_date"] = pd.to_datetime(transaction["posted_date"])

# Merge target onto transactions
tx = transaction.merge(
    consumer[["prism_consumer_id", "evaluation_date", "DQ_TARGET"]],
    on="prism_consumer_id",
    how="inner"
)

#last 90 days
tx = tx[
    (tx["posted_date"] < tx["evaluation_date"]) &
    (tx["posted_date"] >= tx["evaluation_date"] - pd.Timedelta(days=90))
]

# credit is "+" debit is "-"
tx["signed_amount"] = np.where(
    tx["credit_or_debit"] == "CREDIT",
    tx["amount"],
    -tx["amount"]
)

# Net cash flow per consumer
net_cf = (
    tx.groupby("prism_consumer_id")["signed_amount"]
    .sum()
    .reset_index(name="net_cash_flow_last_90d")
)

model_df = consumer.merge(net_cf, on="prism_consumer_id", how="left")

# Train model
train = model_df.dropna(subset=["DQ_TARGET"])

X = train[["net_cash_flow_last_90d"]].fillna(0)
y = train["DQ_TARGET"]

In [27]:
#fit model
model = LogisticRegression()
model.fit(X, y)

# Probability of BAD (DQ_TARGET = 1)
pred_prob_bad = model.predict_proba(X)[:, 1]

# Binary prediction
pred_bad = (pred_prob_bad > 0.5).astype(int)

In [28]:
#auc score
auc = roc_auc_score(y, pred_prob_bad)
auc

0.5508340714309738

In [29]:
#accuracy score
accuracy = accuracy_score(y, pred_bad)
accuracy

0.9161666666666667

In [30]:
#coefficient (negative coefficient means higher cash flow reduces risk of default)
model.coef_

array([[-9.22411161e-06]])